# Marine monitoring observations and site metadata from Basque Country estuaries and coastal waters, 1995–2014 Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant URL**: `https://sen.science/doi/10.71728/senscience.y2wp-d5k2/fair2.json`

This notebook emphasizes a metadata-first workflow:
- load the Croissant dataset,
- inspect metadata safely,
- discover record sets and fields by their `@id`,
- attempt record loading dynamically,
- convert loaded records into pandas DataFrames,
- perform generic exploratory processing that adapts to the available schema.

> Important: all dataset entities should be referenced by their `@id` values. In practice, this notebook first inspects the metadata to discover those IDs dynamically before loading records.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

We keep the dataset URL in a variable so the notebook can be reused easily with other Croissant datasets.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y2wp-d5k2/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))
print('Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Version:', getattr(metadata, 'version', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))
print('Temporal coverage:', getattr(metadata, 'temporal_coverage', getattr(metadata, 'temporalCoverage', 'N/A')))
print('Spatial coverage:', getattr(metadata, 'spatial_coverage', getattr(metadata, 'spatialCoverage', 'N/A')))

In [ ]:
# Convert metadata to a plain JSON-like object for schema introspection.
# We keep `metadata` itself as a single object and inspect a serialized form separately.
metadata_json = metadata.to_json()

print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review available record sets, fields, distributions, and their IDs.

This dataset package description indicates a rich scientific metadata layer. The notebook below explores the machine-readable schema to discover any available Croissant `recordSet` entries and their `field` definitions.

Because Croissant datasets can vary in structure, the next cells are defensive and dynamic.

In [ ]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

record_sets_meta = as_list(metadata_json.get('recordSet', []))
distributions_meta = as_list(metadata_json.get('distribution', []))

print(f'Number of record sets found in metadata JSON: {len(record_sets_meta)}')
print(f'Number of distributions found in metadata JSON: {len(distributions_meta)}')

In [ ]:
# Summarize available distributions by @id where present.
distribution_summary = []
for item in distributions_meta:
    if isinstance(item, dict):
        distribution_summary.append({
            '@id': item.get('@id'),
            '@type': item.get('@type'),
            'name': item.get('name'),
            'contentUrl': item.get('contentUrl'),
            'encodingFormat': item.get('encodingFormat')
        })
    else:
        distribution_summary.append({'value': item})

distribution_df = pd.DataFrame(distribution_summary)
distribution_df

In [ ]:
# Inspect record sets and fields using their @id values.
record_set_rows = []
field_rows = []

for rs in record_sets_meta:
    if not isinstance(rs, dict):
        continue
    record_set_id = rs.get('@id')
    record_set_name = rs.get('name')
    record_set_desc = rs.get('description')
    fields = as_list(rs.get('field', []))

    record_set_rows.append({
        'record_set_@id': record_set_id,
        'record_set_name': record_set_name,
        'record_set_description': record_set_desc,
        'field_count': len(fields)
    })

    for field in fields:
        if not isinstance(field, dict):
            continue
        source = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        extract = source.get('extract', {}) if isinstance(source.get('extract', {}), dict) else {}
        field_rows.append({
            'record_set_@id': record_set_id,
            'field_@id': field.get('@id'),
            'field_name': field.get('name'),
            'description': field.get('description'),
            'dataType': field.get('dataType'),
            'column': source.get('column'),
            'fileObject': source.get('fileObject'),
            'extract_jsonPath': extract.get('jsonPath')
        })

record_sets_df = pd.DataFrame(record_set_rows)
fields_df = pd.DataFrame(field_rows)

print('Record set overview:')
display(record_sets_df)

print('Field overview (first 20 rows):')
display(fields_df.head(20))

### Direct metadata sampling
The template includes iterating through records from a record set. Since this dataset may or may not expose record sets directly in the current schema, the next cell demonstrates how to attempt loading records using discovered `@id` values.

In [ ]:
record_set_ids = record_sets_df['record_set_@id'].dropna().tolist() if not record_sets_df.empty else []
print('Discovered record set @id values:')
print(record_set_ids)

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f'Attempting to preview records from record set @id: {example_record_set_id}')
    try:
        for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print('Could not iterate records for this record set.')
        print(type(e).__name__, str(e))
else:
    print('No record sets were declared in metadata_json["recordSet"].')
    print('This means record loading must be treated as unavailable unless the schema is updated or exposes record sets elsewhere.')

## 3. Data Extraction
Load data from each discovered record set into a DataFrame for analysis.

This section follows the template, but remains robust when a dataset has zero discoverable record sets.

In [ ]:
# Extract data from each record set discovered by @id
record_sets = record_set_ids
dataframes = {}
record_loading_log = []

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        record_loading_log.append({
            'record_set_@id': record_set_id,
            'status': 'loaded',
            'rows': len(dataframes[record_set_id]),
            'columns': len(dataframes[record_set_id].columns)
        })
    except Exception as e:
        record_loading_log.append({
            'record_set_@id': record_set_id,
            'status': 'failed',
            'rows': None,
            'columns': None,
            'error_type': type(e).__name__,
            'error_message': str(e)
        })

loading_log_df = pd.DataFrame(record_loading_log)
loading_log_df

In [ ]:
if dataframes:
    first_loaded_record_set_id = next(iter(dataframes.keys()))
    print('Using loaded record set @id:', first_loaded_record_set_id)
    print('Columns:')
    print(dataframes[first_loaded_record_set_id].columns.tolist())
    display(dataframes[first_loaded_record_set_id].head())
else:
    print('No record sets were successfully loaded into DataFrames.')
    print('We will continue with metadata-level exploration and schema-aware examples.')

### Schema-driven field catalog
Even if records are not directly exposed, Croissant metadata remains useful. The next cell builds a field catalog that can support downstream mapping, validation, and future ingestion workflows.

In [ ]:
if not fields_df.empty:
    display(fields_df.sort_values(['record_set_@id', 'field_@id']).reset_index(drop=True))
else:
    print('No field definitions were found in the declared record sets.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using whichever record set is available.

Since Croissant datasets vary greatly, this section dynamically:
1. selects the first successfully loaded record set,
2. detects numeric and categorical columns,
3. filters a numeric field using a percentile-based threshold,
4. normalizes the selected numeric field,
5. optionally groups by a categorical field.

If no records are available, the notebook performs metadata-level EDA instead.

In [ ]:
def choose_analysis_dataframe(dataframes_dict):
    for rs_id, df in dataframes_dict.items():
        if isinstance(df, pd.DataFrame) and not df.empty:
            return rs_id, df.copy()
    return None, None

analysis_record_set_id, analysis_df = choose_analysis_dataframe(dataframes)

if analysis_df is not None:
    print('Selected record set for EDA:', analysis_record_set_id)
    print('Shape:', analysis_df.shape)
    display(analysis_df.head())
else:
    print('No loaded tabular data available for record-level EDA.')

In [ ]:
if analysis_df is not None:
    # Try to coerce object columns to numeric where appropriate
    coerced_df = analysis_df.copy()
    for col in coerced_df.columns:
        if coerced_df[col].dtype == 'object':
            coerced_candidate = pd.to_numeric(coerced_df[col], errors='ignore')
            coerced_df[col] = coerced_candidate

    numeric_columns = coerced_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = [
        c for c in coerced_df.columns
        if c not in numeric_columns and coerced_df[c].dtype == 'object'
    ]

    print('Numeric columns:', numeric_columns)
    print('Categorical columns:', categorical_columns)
else:
    numeric_columns = []
    categorical_columns = []

In [ ]:
if analysis_df is not None and numeric_columns:
    # Select a numeric field for analysis using the actual column name, ideally matching a field @id if present.
    numeric_field = numeric_columns[0]
    numeric_series = pd.to_numeric(coerced_df[numeric_field], errors='coerce').dropna()

    if len(numeric_series) > 0:
        threshold = numeric_series.quantile(0.75)
        filtered_df = coerced_df[pd.to_numeric(coerced_df[numeric_field], errors='coerce') > threshold].copy()
        print(f'Filtered records with {numeric_field} > {threshold}:')
        display(filtered_df.head())

        std = filtered_df[numeric_field].std()
        if pd.notna(std) and std != 0:
            filtered_df[f'{numeric_field}_normalized'] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / std
        else:
            filtered_df[f'{numeric_field}_normalized'] = 0.0

        print(f'Normalized {numeric_field} for filtered records:')
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        group_field = categorical_columns[0] if categorical_columns else None
        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame('mean_value')
            print(f'Grouped data by {group_field}:')
            display(grouped_df.head())
        else:
            print('No suitable categorical group field was available for grouping.')
    else:
        print('The selected numeric field has no usable numeric values after coercion.')
else:
    print('Skipping record-level filtering and normalization because no numeric columns were available.')

In [ ]:
# Metadata-level EDA fallback
print('Metadata-level summaries')
print('- Number of distributions:', len(distributions_meta))
print('- Number of declared record sets:', len(record_sets_meta))
print('- Number of declared fields:', len(fields_df))

keyword_list = as_list(metadata_json.get('keywords', []))
use_case_list = as_list(metadata_json.get('dataUseCases', []))
bias_list = as_list(metadata_json.get('dataBiases', []))

print('- Number of keywords:', len(keyword_list))
print('- Number of documented use cases:', len(use_case_list))
print('- Number of documented biases:', len(bias_list))

## 5. Visualization
Visualize available metadata and, when possible, record-level distributions or relationships between fields.

In [ ]:
# Visualization 1: field counts by record set (metadata-level)
if not record_sets_df.empty:
    plt.figure(figsize=(10, 5))
    plot_df = record_sets_df.copy()
    plot_df['label'] = plot_df['record_set_name'].fillna(plot_df['record_set_@id'])
    sns.barplot(data=plot_df, x='field_count', y='label', palette='viridis')
    plt.title('Declared field count by record set')
    plt.xlabel('Field count')
    plt.ylabel('Record set')
    plt.tight_layout()
    plt.show()
else:
    print('No record sets available for field-count visualization.')

In [ ]:
# Visualization 2: keyword frequency display (simple bar chart of listed keywords)
if keyword_list:
    keyword_df = pd.DataFrame({'keyword': keyword_list, 'count': 1})
    plt.figure(figsize=(10, max(4, len(keyword_df) * 0.25)))
    sns.barplot(data=keyword_df, x='count', y='keyword', color='steelblue')
    plt.title('Dataset keywords')
    plt.xlabel('Presence')
    plt.ylabel('Keyword')
    plt.tight_layout()
    plt.show()
else:
    print('No keywords were available for visualization.')

In [ ]:
# Visualization 3: record-level numeric distribution, if available
if analysis_df is not None and numeric_columns:
    numeric_field = numeric_columns[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(coerced_df[numeric_field], errors='coerce').dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric record-level field was available for histogram plotting.')

In [ ]:
# Visualization 4: top categories for a categorical field, if available
if analysis_df is not None and categorical_columns:
    group_field = categorical_columns[0]
    top_counts = coerced_df[group_field].astype(str).value_counts(dropna=False).head(15).reset_index()
    top_counts.columns = [group_field, 'count']
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_counts, x='count', y=group_field, palette='magma')
    plt.title(f'Top categories in {group_field}')
    plt.xlabel('Count')
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()
else:
    print('No categorical record-level field was available for category visualization.')

## 6. Conclusion
This notebook demonstrated how to explore a Croissant dataset using `mlcroissant` in a schema-aware way.

### Key takeaways
- The dataset metadata loads successfully from the provided Croissant URL.
- Metadata inspection can be performed safely by keeping `dataset.metadata` as a single object and serializing it only when needed for schema introspection.
- Record sets, fields, and distributions should be referenced using their `@id` values.
- The notebook dynamically attempts record loading and gracefully handles schemas where record sets are not currently exposed.
- Even when record-level extraction is unavailable, Croissant metadata still supports useful exploration, validation, documentation, and planning for downstream analysis.

### Suggested next steps
- Review the dataset's distributions and source files to identify which data assets can be connected to Croissant record sets.
- If the schema is updated to include populated `recordSet` entries, rerun this notebook to automatically load and analyze records.
- Extend the EDA section with domain-specific analyses for station, parameter, depth, taxon, and time-series structure once tabular records are exposed.